# Part 1: Financial News Data Integrity, Preprocessing, and Relevance Analysis

## AI-Based Stock Analyzer — Financial News Sentiment Analysis

**Course:** Data Science Tools and Techniques  
**Project Phase:** Part 1 — News Dataset Acquisition, Data Integrity Verification, Preprocessing, Text Cleaning, and Ticker Relevance Analysis  
**Dataset Source:** FNSPID (Financial News and Stock Price Integration Dataset)  
**Historical Period:** 2015-01-02 to 2023-12-16  
**Author / Analysis Lead:** Tayyab Akhtar (Financial News Preprocessing & Sentiment Analysis)  
**Team Members:** Sikander (EDA, Time-Series Diagnostics & Statistical Analysis), Hamza (Data Cleaning & Feature Engineering)

---

### Project Objectives & Core Questions

1. **Data Integrity & Validation:** Examine missing values, duplicate records, publication dates, and ticker distributions.
2. **Missing Article Investigation:** Identify missing article sources and evaluate whether the original content can be recovered.
3. **Data Cleaning:** Remove unusable records, redundant columns, and duplicate article content while preserving meaningful financial information.
4. **Timestamp Standardization:** Convert publication timestamps into consistent daily dates for alignment with historical stock prices.
5. **Financial Text Preprocessing:** Remove HTML tags, normalize whitespace, and preserve financial terminology, numerical values, and ticker symbols.
6. **Ticker Relevance Analysis:** Evaluate whether articles explicitly mention their assigned companies and identify potentially irrelevant news.
7. **Preparation for FinBERT:** Prepare cleaned financial articles for sentence segmentation, relevance filtering, and sentiment classification.
8. **Integration with Predictive Models:** Prepare sentiment features for combination with LSTM–GRU stock price predictions and Mistral 7B financial analysis.

### 1. Environment Setup

In this section, we import the required Python libraries, including Pandas, NumPy, Regular Expressions, BeautifulSoup, and supporting text-processing utilities. These tools are used for dataset inspection, missing-value analysis, duplicate detection, timestamp normalization, and financial news cleaning.

In [2]:

import pandas as pd

# Load raw dataset
df = pd.read_csv(r"C:\Users\LENOVO\OneDrive\Documents\DataScience\DSTT Project\filtered_data\filtered_financial_news.csv")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nMissing percentages:")
print((df.isna().mean() * 100).round(2))

print("\nTicker distribution:")
print(df["ticker"].value_counts(dropna=False))

print("\nExact duplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate URLs (non-null):")
print(df.loc[df["url"].notna(), "url"].duplicated().sum())

print("\nDate samples:")
print(df["date"].head(10).tolist())

print("\nDate range (raw values):")
print(df["date"].dropna().astype(str).agg(["min", "max"]))

print("\nSample news:")
print(df[["date", "ticker", "title"]].head(3).to_string())


Dataset shape: (49921, 7)

Column names:
['date', 'title', 'ticker', 'url', 'publisher', 'author', 'article']

Data types:
date             str
title            str
ticker           str
url              str
publisher        str
author       float64
article          str
dtype: object

Missing values:
date             0
title            0
ticker           0
url              0
publisher    40798
author       49921
article       9124
dtype: int64

Missing percentages:
date           0.00
title          0.00
ticker         0.00
url            0.00
publisher     81.73
author       100.00
article       18.28
dtype: float64

Ticker distribution:
ticker
NVDA    11191
WFC     10483
MRK      8465
SPY      7225
OXY      6526
COST     6031
Name: count, dtype: int64

Exact duplicate rows:
0

Duplicate URLs (non-null):
705

Date samples:
['2015-01-02 00:00:00+00:00', '2015-01-03 00:00:00+00:00', '2015-01-05 00:00:00+00:00', '2015-01-06 00:00:00+00:00', '2015-01-06 00:00:00+00:00', '2015-01-06 00:00:0

In [3]:

for col in ["title", "article", "url", "publisher", "author"]:
    missing = (
        df[col].isna()
        | df[col].astype("string").str.strip().eq("").fillna(False)
    )
    print(f"{col}: {missing.sum()} missing/empty ({missing.mean()*100:.2f}%)")

print("\nRows with headline but no article:")
title_valid = df["title"].notna() & df["title"].astype("string").str.strip().ne("").fillna(False)
article_valid = df["article"].notna() & df["article"].astype("string").str.strip().ne("").fillna(False)

print((title_valid & ~article_valid).sum())


title: 0 missing/empty (0.00%)
article: 9124 missing/empty (18.28%)
url: 0 missing/empty (0.00%)
publisher: 40798 missing/empty (81.73%)
author: 49921 missing/empty (100.00%)

Rows with headline but no article:
9124


Handle Missing Values
The dataset contains 49,921 news records, with 9,124 (18.28%) missing article bodies. We decided to remove these records because our analysis requires full article text. The author column is entirely missing, while publisher is 81.73% missing, so both will be dropped. This leaves 40,797 articles for further preprocessing, while the original dataset is preserved.

In [4]:
from urllib.parse import urlparse
import pandas as pd

# Work on a copy
news_df = df.copy()

# Drop completely empty column
news_df = news_df.drop(columns=["author"])

# Extract source domain
def extract_domain(url):
    try:
        domain = urlparse(str(url)).netloc.lower()
        return domain.removeprefix("www.") or None
    except (ValueError, TypeError):
        return None

news_df["source_domain"] = news_df["url"].apply(extract_domain)

# Identify missing article bodies
missing_mask = (
    news_df["article"].isna()
    | news_df["article"].astype("string").str.strip().eq("").fillna(False)
)

missing_news = news_df.loc[missing_mask].copy()

print("Missing articles:", len(missing_news))

print("\nTop 20 domains with missing articles:")
print(missing_news["source_domain"].value_counts().head(20))

print("\nMissing articles by ticker:")
print(missing_news["ticker"].value_counts())

print("\nMissing article URL samples:")
for url in missing_news["url"].head(15):
    print(url)

print("\nInvalid or unrecognized URL domains:")
print(news_df["source_domain"].isna().sum())


Missing articles: 9124

Top 20 domains with missing articles:
source_domain
benzinga.com          7011
seekingalpha.com      1188
zacks.com              409
gurufocus.com          362
talkmarkets.com         59
investors.com           56
news.investors.com      17
marketfy.com            16
vetr.com                 4
foxbusiness.com          1
nasdaq.com               1
Name: count, dtype: int64

Missing articles by ticker:
ticker
OXY     2917
NVDA    2475
MRK     1923
WFC     1794
SPY       15
Name: count, dtype: int64

Missing article URL samples:
https://www.benzinga.com/news/15/01/5117771/bernstein-upgrades-merck-to-outperform
https://www.benzinga.com/news/15/01/5122671/fda-has-approved-mercks-request-to-supplement-pneumovax-bla-will-add-2d-barcode-o
https://www.benzinga.com/trading-ideas/long-ideas/15/01/5136442/biotechs-presenting-at-the-j-p-morgan-healthcare-conference-d
https://www.benzinga.com/analyst-ratings/analyst-color/15/01/5135725/wall-streets-hall-of-fame-and-hall-of-sh

Analysis of Missing Articles
We analyzed the 9,124 missing articles by source domain and ticker to identify patterns and potential recovery options. Most missing articles originated from Benzinga (7,011), followed by Seeking Alpha (1,188). We also examined article URLs to determine whether the missing content could be retrieved from the original websites.

In [5]:

import trafilatura
import requests
import pandas as pd
import time

# Test URLs across different publishers
test_news = (
    missing_news.groupby("source_domain", group_keys=False)
    .head(2)
    .head(10)
    .copy()
)

def extract_article(url):
    try:
        response = requests.get(
            url,
            headers={
                "User-Agent": "StockNewsResearchBot/1.0"
            },
            timeout=15,
            allow_redirects=True
        )

        if response.status_code != 200:
            return None, f"HTTP_{response.status_code}"

        content = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            favor_precision=True
        )

        if not content or len(content.strip()) < 100:
            return None, "NO_CONTENT"

        return content, "SUCCESS"

    except requests.RequestException as e:
        return None, type(e).__name__

results = []

for idx, row in test_news.iterrows():
    print(f"Testing: {row['source_domain']}")

    article_text, status = extract_article(row["url"])

    results.append({
        "index": idx,
        "domain": row["source_domain"],
        "status": status,
        "length": len(article_text) if article_text else 0,
        "preview": article_text[:200] if article_text else None
    })

    time.sleep(2)

recovery_test = pd.DataFrame(results)

display(recovery_test)


Testing: benzinga.com
Testing: benzinga.com
Testing: gurufocus.com
Testing: zacks.com
Testing: seekingalpha.com
Testing: seekingalpha.com
Testing: zacks.com
Testing: marketfy.com
Testing: news.investors.com
Testing: news.investors.com


,index,domain,status,length,preview
0,6032,benzinga.com,SUCCESS,365,Never miss a trade again with the fastest news...
1,6037,benzinga.com,HTTP_403,0,NaN
2,11192,gurufocus.com,HTTP_403,0,NaN
3,11202,zacks.com,SUCCESS,2156,"This page has not been authorized, sponsored, ..."
4,25687,seekingalpha.com,HTTP_403,0,NaN
5,25690,seekingalpha.com,HTTP_403,0,NaN
6,25710,zacks.com,SUCCESS,2156,"This page has not been authorized, sponsored, ..."
7,25718,marketfy.com,HTTP_403,0,NaN
8,25732,news.investors.com,HTTP_404,0,NaN
9,25737,news.investors.com,HTTP_404,0,NaN


Attempt to Recover Missing Articles
We attempted to retrieve missing article content from original news websites using web scraping. Most requests failed due to HTTP 403 (access denied) or HTTP 404 (page not found) errors. Some requests succeeded but returned irrelevant website content instead of actual articles, making recovery unreliable.

In [11]:

import requests
import certifi

url = missing_news.loc[
    missing_news["source_domain"] == "benzinga.com",
    "url"
].iloc[0]

try:
    response = requests.get(
        url,
        timeout=15,
        verify=certifi.where(),
        headers={"User-Agent": "Mozilla/5.0"}
    )

    print("Status:", response.status_code)
    print("Final URL:", response.url)
    print("Content length:", len(response.content))

except requests.exceptions.SSLError as e:
    print("SSL verification failed:")
    print(str(e)[:1000])

except requests.exceptions.RequestException as e:
    print("Request failed:", str(e)[:1000])


Status: 403
Final URL: https://www.benzinga.com/news/15/01/5117771/bernstein-upgrades-merck-to-outperform
Content length: 5661


Investigating Benzinga Access
We specifically tested Benzinga because it accounted for 7,011 of the 9,124 missing articles (76.84%). We checked whether the original article could be accessed and its content recovered. The server returned HTTP 403 (Forbidden), indicating that automated access was blocked despite receiving 5,661 bytes of response content.

The 5,661 bytes likely contain an error or access-denied page, not the article itself. Therefore, we could not reliably recover the missing Benzinga content.

In [12]:

# Preserve original dataset
raw_news_df = df.copy()

# Remove completely empty author column
news_df = raw_news_df.drop(columns=["author"]).copy()

# Remove records with missing or empty article bodies
valid_article = (
    news_df["article"].notna()
    & news_df["article"].astype("string").str.strip().ne("").fillna(False)
)

news_df = news_df.loc[valid_article].copy()

# Reset index
news_df.reset_index(drop=True, inplace=True)

print("Original records:", len(raw_news_df))
print("Removed records:", len(raw_news_df) - len(news_df))
print("Remaining records:", len(news_df))

print("\nRemaining missing values:")
print(news_df.isna().sum())

print("\nTicker distribution:")
print(news_df["ticker"].value_counts())

# Save intermediate dataset
news_df.to_csv(
    "stock_news_filtered.csv",
    index=False
)

print("\nFiltered dataset saved successfully.")


Original records: 49921
Removed records: 9124
Remaining records: 40797

Remaining missing values:
date             0
title            0
ticker           0
url              0
publisher    40797
article          0
dtype: int64

Ticker distribution:
ticker
NVDA    8716
WFC     8689
SPY     7210
MRK     6542
COST    6031
OXY     3609
Name: count, dtype: int64

Filtered dataset saved successfully.


Decision: Remove Records with Missing Articles
We removed 9,124 records (18.28%) because their article bodies were missing and attempts to recover them through web scraping were unsuccessful. Since our FinBERT pipeline will analyze article content, retaining these records would provide incomplete inputs. This left 40,797 complete articles, although removing them may introduce selection bias and reduce news coverage

In [13]:

from urllib.parse import urlparse

# Check publisher coverage after article filtering
print("Publisher missing percentage:",
      round(news_df["publisher"].isna().mean() * 100, 2))

# Extract website domain
def extract_domain(url):
    try:
        domain = urlparse(str(url)).hostname
        return domain.lower().removeprefix("www.") if domain else None
    except (ValueError, TypeError):
        return None

news_df["source_domain"] = news_df["url"].apply(extract_domain)

# Check extracted domains
print("\nTop 15 source domains:")
print(news_df["source_domain"].value_counts().head(15))

print("\nMissing source domains:")
print(news_df["source_domain"].isna().sum())

# Remove publisher column
news_df.drop(columns=["publisher"], inplace=True)

print("\nFinal columns:")
print(news_df.columns.tolist())


Publisher missing percentage: 100.0

Top 15 source domains:
source_domain
nasdaq.com    40797
Name: count, dtype: int64

Missing source domains:
0

Final columns:
['date', 'title', 'ticker', 'url', 'article', 'source_domain']


In [14]:
news_df.drop(
    columns=["publisher", "source_domain"],
    errors="ignore",
    inplace=True
)

print("Final columns:", news_df.columns.tolist())
print("Dataset shape:", news_df.shape)

news_df.to_csv("stock_news_filtered.csv", index=False)

Final columns: ['date', 'title', 'ticker', 'url', 'article']
Dataset shape: (40797, 5)


Publisher and Source Domain Analysis
We extracted source domains from article URLs to identify news publishers. All 40,797 remaining articles originated from nasdaq.com, while the publisher column was 100% missing. Therefore, we decided to remove both publisher and source_domain because they provided no useful additional information

In [15]:
import pandas as pd

# Parse timestamps
parsed_dates = pd.to_datetime(
    news_df["date"],
    utc=True,
    errors="coerce"
)

print("Invalid timestamps:", parsed_dates.isna().sum())

# Check how many timestamps contain non-zero time
non_midnight = (
    parsed_dates.notna()
    & (
        (parsed_dates.dt.hour != 0)
        | (parsed_dates.dt.minute != 0)
        | (parsed_dates.dt.second != 0)
    )
)

print("Non-midnight timestamps:", non_midnight.sum())

print("\nSample non-midnight timestamps:")
print(news_df.loc[non_midnight, "date"].head(10))


Invalid timestamps: 0
Non-midnight timestamps: 35

Sample non-midnight timestamps:
6028     2023-12-16 16:00:00+00:00
6029     2023-12-16 17:00:00+00:00
6030     2023-12-16 21:00:00+00:00
12569    2023-12-16 01:00:00+00:00
12570    2023-12-16 04:00:00+00:00
12571    2023-12-16 08:00:00+00:00
12572    2023-12-16 10:00:00+00:00
21273    2023-12-16 12:00:00+00:00
21274    2023-12-16 14:00:00+00:00
21275    2023-12-16 14:00:00+00:00
Name: date, dtype: str


In [16]:
import pandas as pd

# Convert to calendar date using the recorded UTC date
news_df["date"] = pd.to_datetime(
    news_df["date"],
    utc=True,
    errors="raise"
).dt.strftime("%Y-%m-%d")

print("Dataset shape:", news_df.shape)

print("\nDate range:")
print(news_df["date"].min(), "to", news_df["date"].max())

print("\nSample:")
print(news_df[["date", "ticker", "title"]].head())

# Save intermediate cleaned dataset
news_df.to_csv("stock_news_filtered.csv", index=False)

print("\nDate conversion completed.")


Dataset shape: (40797, 5)

Date range:
2015-01-02 to 2023-12-16

Sample:
         date ticker                                              title
0  2015-01-02   COST  Signature Select Pursues Financial Services St...
1  2015-01-03   COST  Did Your Favorite Retailer Make Its Christmas ...
2  2015-01-05   COST  One Put, One Call Option To Know About for Cos...
3  2015-01-06   COST  Costco Sustains Sales Momentum Amid Stiff Comp...
4  2015-01-06   COST  Nordstrom's Second Louisiana Rack Store to Ope...

Date conversion completed.


Timestamp Validation and Standardization
We validated all article timestamps and found no invalid dates. Only 35 records contained non-midnight timestamps, while the remaining 40,762 were recorded at midnight. Since our stock price dataset uses daily data, we decided to standardize news timestamps to YYYY-MM-DD and align articles with the next trading session to reduce look-ahead bias.

In [17]:
import pandas as pd

print("Total records:", len(news_df))

# 1. Exact duplicates
exact_duplicates = news_df.duplicated().sum()

# 2. Same URL and ticker
url_ticker_duplicates = news_df.duplicated(
    subset=["url", "ticker"]
).sum()

# 3. Same title, ticker and date
title_ticker_date_duplicates = news_df.duplicated(
    subset=["title", "ticker", "date"]
).sum()

# 4. Same URL across different tickers
multi_ticker_urls = (
    news_df.groupby("url")["ticker"]
    .nunique()
)

multi_ticker_urls = multi_ticker_urls[
    multi_ticker_urls > 1
]

# 5. Same title across different dates
multi_date_titles = (
    news_df.groupby(["ticker", "title"])["date"]
    .nunique()
)

multi_date_titles = multi_date_titles[
    multi_date_titles > 1
]

print("\nExact duplicate rows:", exact_duplicates)
print("Duplicate URL + ticker:", url_ticker_duplicates)
print("Duplicate title + ticker + date:",
      title_ticker_date_duplicates)

print("URLs associated with multiple tickers:",
      len(multi_ticker_urls))

print("Ticker-title pairs appearing on multiple dates:",
      len(multi_date_titles))

# 6. Show examples of repeated URL + ticker
print("\nSample duplicate URL + ticker records:")

duplicates = news_df[
    news_df.duplicated(
        subset=["url", "ticker"],
        keep=False
    )
].sort_values(["url", "ticker"])

display(
    duplicates[
        ["date", "ticker", "title", "url"]
    ].head(20)
)

# 7. Show examples of duplicate titles on same date
print("\nSample duplicate title + ticker + date:")

title_duplicates = news_df[
    news_df.duplicated(
        subset=["title", "ticker", "date"],
        keep=False
    )
].sort_values(["ticker", "date", "title"])

display(
    title_duplicates[
        ["date", "ticker", "title", "url"]
    ].head(20)
)


Total records: 40797

Exact duplicate rows: 0
Duplicate URL + ticker: 0
Duplicate title + ticker + date: 518
URLs associated with multiple tickers: 602
Ticker-title pairs appearing on multiple dates: 1252

Sample duplicate URL + ticker records:


,date,ticker,title,url



Sample duplicate title + ticker + date:


,date,ticker,title,url
164,2015-05-26,COST,Will Costco (COST) Let Down Investors this Ear...,https://www.nasdaq.com/articles/will-costco-co...
165,2015-05-26,COST,Will Costco (COST) Let Down Investors this Ear...,https://www.nasdaq.com/articles/will-costco-co...
372,2015-12-04,COST,Weak Q3 Earnings Season Now Behind Us,https://www.nasdaq.com/articles/weak-q3-earnin...
373,2015-12-04,COST,Weak Q3 Earnings Season Now Behind Us,https://www.nasdaq.com/articles/weak-q3-earnin...
583,2016-05-26,COST,Not Even $50 Oil Takes The Edge Off Nervous Ma...,https://www.nasdaq.com/articles/not-even-50-oi...
584,2016-05-26,COST,Not Even $50 Oil Takes The Edge Off Nervous Ma...,https://www.nasdaq.com/articles/not-even-50-oi...
621,2016-06-17,COST,What Will the Q2 Earnings Season Bring?,https://www.nasdaq.com/articles/what-will-q2-e...
622,2016-06-17,COST,What Will the Q2 Earnings Season Bring?,https://www.nasdaq.com/articles/what-will-q2-e...
832,2016-11-16,COST,Wal-Mart's Q3 2016 Earnings Preview: What To E...,https://www.nasdaq.com/articles/wal-marts-q3-2...
833,2016-11-16,COST,Wal-Mart's Q3 2016 Earnings Preview: What To E...,https://www.nasdaq.com/articles/wal-marts-q3-2...


Duplicate Records Analysis
We checked for duplicate articles using URLs, titles, tickers, and publication dates. Although no exact duplicates were found, 518 repeated title–ticker–date occurrences required further investigation. We retained articles associated with multiple tickers or published on different dates, as these may represent valid news records.

In [18]:

# Find repeated headlines for the same ticker and date
duplicate_headlines = news_df[
    news_df.duplicated(
        subset=["date", "ticker", "title"],
        keep=False
    )
].copy()

# Group duplicate headlines and compare article bodies
duplicate_summary = (
    duplicate_headlines
    .groupby(["date", "ticker", "title"])
    .agg(
        total_records=("article", "size"),
        unique_articles=("article", "nunique"),
        unique_urls=("url", "nunique")
    )
    .reset_index()
)

print("Duplicate headline groups:", len(duplicate_summary))

print("\nGroups with identical article bodies:")
print((duplicate_summary["unique_articles"] == 1).sum())

print("\nGroups with different article bodies:")
print((duplicate_summary["unique_articles"] > 1).sum())

print("\nGroups with different URLs:")
print((duplicate_summary["unique_urls"] > 1).sum())

print("\nSample groups with different article bodies:")
display(
    duplicate_summary[
        duplicate_summary["unique_articles"] > 1
    ].head(10)
)


Duplicate headline groups: 508

Groups with identical article bodies:
119

Groups with different article bodies:
389

Groups with different URLs:
508

Sample groups with different article bodies:


,date,ticker,title,total_records,unique_articles,unique_urls
2,2016-01-08,WFC,China & the Q4 Earnings Season,2,2,2
3,2016-05-26,COST,Not Even $50 Oil Takes The Edge Off Nervous Ma...,2,2,2
8,2016-12-29,COST,Will the New Year Bring Earnings Growth?,2,2,2
9,2017-01-18,WFC,Bank Earnings & the Trump Effect,2,2,2
12,2017-06-12,WFC,"Financial Sector Update for 06/12/2017: JPM, B...",2,2,2
15,2017-10-02,COST,Walmart to Roll Out Private-Label Brand to Tak...,2,2,2
18,2018-01-16,WFC,4 Blue-Chip Stocks to Buy in Q4 Earnings,2,2,2
19,2018-01-17,WFC,5 Stocks to Fall Back On Amid Lurking Governme...,2,2,2
20,2018-01-18,SPY,Q4 Earnings Season Scorecard,2,2,2
21,2018-01-22,MRK,Will These 4 Large-Cap Drug Stocks Be Big Winn...,2,2,2


Duplicate Headline Investigation
We examined 508 groups with repeated headlines and found that 119 had identical article bodies, while 389 contained different content. Since identical headlines do not necessarily indicate duplicate articles, we decided to remove records only when the ticker, date, and article body matched, preserving articles with different content.

In [19]:
import re

# Normalize text only for duplicate comparison
def normalize_for_dedup(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

news_df["article_dedup_key"] = (
    news_df["article"].apply(normalize_for_dedup)
)

before = len(news_df)

# Remove identical article bodies for same ticker and date
news_df = news_df.drop_duplicates(
    subset=["date", "ticker", "article_dedup_key"],
    keep="first"
).copy()

removed = before - len(news_df)

# Remove temporary comparison column
news_df.drop(columns=["article_dedup_key"], inplace=True)

news_df.reset_index(drop=True, inplace=True)

print("Records before:", before)
print("Duplicates removed:", removed)
print("Records remaining:", len(news_df))

print("\nRemaining duplicates by ticker/date/article:")
print(
    news_df.duplicated(
        subset=["date", "ticker", "article"]
    ).sum()
)

# Save intermediate dataset
news_df.to_csv("stock_news_deduplicated.csv", index=False)


Records before: 40797
Duplicates removed: 146
Records remaining: 40651

Remaining duplicates by ticker/date/article:
0


Duplicate Removal
We removed 146 duplicate records based on matching ticker, publication date, and article content to prevent repeated news from influencing sentiment analysis. The dataset was reduced from 40,797 to 40,651 articles, with no remaining duplicates under these criteria.

In [20]:

import pandas as pd
import re

# Article and title lengths
news_df["article_length"] = news_df["article"].astype(str).str.len()
news_df["title_length"] = news_df["title"].astype(str).str.len()

print("Article length statistics:")
print(news_df["article_length"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nTitle length statistics:")
print(news_df["title_length"].describe())

# HTML tag detection
html_pattern = r"<[a-zA-Z][^>]*>"

html_count = news_df["article"].str.contains(
    html_pattern,
    regex=True,
    na=False
).sum()

print("\nArticles containing HTML-like tags:", html_count)

# Common HTML entities
entity_count = news_df["article"].str.contains(
    r"&(?:amp|nbsp|quot|lt|gt|#\d+|#x[0-9a-fA-F]+);",
    regex=True,
    na=False
).sum()

print("Articles containing common HTML entities:", entity_count)

# Check ticker values
print("\nUnique tickers:")
print(news_df["ticker"].unique())

# Display short article samples
print("\nSample articles:")

for idx, row in news_df.sample(
    min(5, len(news_df)),
    random_state=42
).iterrows():

    print("\n" + "=" * 60)
    print("Ticker:", row["ticker"])
    print("Title:", row["title"])
    print("Article preview:", str(row["article"])[:600])


Article length statistics:
count     40651.000000
mean       4629.139578
std        4793.107457
min         133.000000
25%        2193.500000
50%        4150.000000
75%        5688.000000
90%        7291.000000
95%        8853.000000
99%       22184.500000
max      105235.000000
Name: article_length, dtype: float64

Title length statistics:
count    40651.000000
mean        58.901774
std         18.653239
min         14.000000
25%         46.000000
50%         57.000000
75%         70.000000
max        234.000000
Name: title_length, dtype: float64

Articles containing HTML-like tags: 34
Articles containing common HTML entities: 0

Unique tickers:
<StringArray>
['COST', 'MRK', 'NVDA', 'OXY', 'SPY', 'WFC']
Length: 6, dtype: str

Sample articles:

Ticker: SPY
Title: AMGN Dividend Yield Pushes Past 3%
Article preview: Looking at the universe of stocks we cover at Dividend Channel , in trading on Thursday, shares of Amgen Inc (Symbol: AMGN) were yielding above the 3% mark based on its quart

Text Quality and Length Analysis
We analyzed article and headline lengths to assess text quality and suitability for FinBERT. Articles averaged 4,629 characters, with some reaching 105,235 characters, potentially exceeding FinBERT's 512-token limit. We also detected 34 articles containing HTML-like tags and identified potential ticker-relevance issues, requiring further cleaning and relevance filtering.

In [21]:

import re
import html
import unicodedata
from bs4 import BeautifulSoup

def clean_financial_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML markup if present
    if re.search(r"<[a-zA-Z][^>]*>", text):
        text = BeautifulSoup(text, "html.parser").get_text(" ")

    # Normalize Unicode presentation forms
    text = unicodedata.normalize("NFC", text)

    # Remove invisible formatting characters
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)

    # Normalize whitespace without changing sentence order
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Preserve original text for auditing
news_df["title_clean"] = news_df["title"].apply(clean_financial_text)
news_df["article_clean"] = news_df["article"].apply(clean_financial_text)

print("Empty cleaned titles:",
      news_df["title_clean"].eq("").sum())

print("Empty cleaned articles:",
      news_df["article_clean"].eq("").sum())

print("\nSample cleaned article:")
print(news_df["article_clean"].iloc[0][:1000])


Empty cleaned titles: 0
Empty cleaned articles: 0

Sample cleaned article:
More than a quarter of Signature Select Canadian Fund ( Trades , Portfolio )'s investments are in financial services, and the firm continued to invest in financial stocks in the third quarter, but its investments in the quarter were diverse. You will find activity in numerous financial services in Signature Select's portfolio in the third quarter, but you will also find other kinds of stocks in Signature Select's portfolio as well. The fund had a 19.21% return in 2013, more than double its rate of return in 2012. Warren Buffett Recent Buys Signature Select added holdings in 21 companies that are new to its portfolio - UBS Group AG ( UBS ), Manulife Financial Corp ( MFC ), Dover Corp ( DOV ), Costco Wholesale Corp ( COST ), Adidas AG (ADS.Germany), Baker Hughes Inc ( BHI ), Mitsubishi Electric Corp (6503.Japan), Estee Lauder Cos Inc ( EL ), Newell Rubbermaid Inc (NWL), Drax Group PLC (DRX.UK), Dollar Tree Stores 

Financial Text Cleaning
We cleaned article bodies and headlines by removing HTML tags, normalizing whitespace, and handling special characters. We preserved financial figures, company names, ticker symbols, and punctuation to maintain important information for FinBERT sentiment analysis. After cleaning, no titles or article bodies were empty.

In [1]:

print("Articles changed:",
      (news_df["article"] != news_df["article_clean"]).sum())

print("Titles changed:",
      (news_df["title"] != news_df["title_clean"]).sum())

print("\nCleaned article length statistics:")
print(news_df["article_clean"].str.len().describe())

print("\nRemaining HTML-like tags:")
print(
    news_df["article_clean"]
    .str.contains(r"<[a-zA-Z][^>]*>", regex=True, na=False)
    .sum()
)


NameError: name 'news_df' is not defined

Text Cleaning Validation
We validated the cleaning process and found that 40,647 articles and 56 titles were modified, primarily through text normalization. No HTML-like tags remained, confirming successful cleaning. All 40,651 articles were preserved, with their financial information intact for subsequent sentiment analysis.

In [23]:

import re
import pandas as pd

# Basic aliases for the six tickers
ticker_aliases = {
    "NVDA": ["NVDA", "NVIDIA"],
    "WFC": ["WFC", "Wells Fargo"],
    "MRK": ["MRK", "Merck"],
    "OXY": ["OXY", "Occidental Petroleum", "Occidental"],
    "COST": ["COST", "Costco"],
    "SPY": ["SPY", "S&P 500", "S&P500", "SPDR S&P"]
}

def check_relevance(row):
    ticker = row["ticker"]
    title = str(row["title_clean"])
    article = str(row["article_clean"])

    aliases = ticker_aliases.get(ticker, [ticker])

    title_match = any(
        re.search(r"\b" + re.escape(alias) + r"\b", title, re.IGNORECASE)
        for alias in aliases
    )

    article_match = any(
        re.search(r"\b" + re.escape(alias) + r"\b", article, re.IGNORECASE)
        for alias in aliases
    )

    return pd.Series({
        "ticker_in_title": title_match,
        "ticker_in_article": article_match
    })

news_df[["ticker_in_title", "ticker_in_article"]] = (
    news_df.apply(check_relevance, axis=1)
)

news_df["relevance_status"] = "No explicit match"

news_df.loc[
    news_df["ticker_in_article"],
    "relevance_status"
] = "Article match"

news_df.loc[
    news_df["ticker_in_title"],
    "relevance_status"
] = "Title match"

print("Relevance status distribution:")
print(news_df["relevance_status"].value_counts())

print("\nRelevance status by ticker:")
print(
    pd.crosstab(
        news_df["ticker"],
        news_df["relevance_status"],
        normalize="index"
    ).round(3)
)

print("\nSample articles without explicit matches:")
display(
    news_df.loc[
        news_df["relevance_status"] == "No explicit match",
        ["date", "ticker", "title_clean"]
    ].head(20)
)


Relevance status distribution:
relevance_status
Article match        26176
Title match          13769
No explicit match      706
Name: count, dtype: int64

Relevance status by ticker:
relevance_status  Article match  No explicit match  Title match
ticker                                                         
COST                      0.540              0.007        0.454
MRK                       0.557              0.032        0.411
NVDA                      0.639              0.014        0.347
OXY                       0.562              0.010        0.428
SPY                       0.873              0.012        0.115
WFC                       0.631              0.025        0.344

Sample articles without explicit matches:


,date,ticker,title_clean
22,2015-02-03,COST,Wal-Mart And China: A Story Of Missing Custome...
103,2015-04-07,COST,What's Driving Wal-Mart's Average Square Foota...
161,2015-05-22,COST,Target On-Target In The First Quarter
393,2015-12-14,COST,A safe way to play one of the Dow's biggest lo...
458,2016-02-24,COST,The Lesson of Investors' Psychology
572,2016-05-23,COST,Warren Buffett’s Worst Stocks of 2016 So Far
657,2016-07-13,COST,Dividend Investing in a Roth IRA
829,2016-11-16,COST,Wal-Mart's Q3 2016 Earnings Preview: What To E...
1049,2017-03-21,COST,What’s the Key to a Long-Term Competitive Adva...
1051,2017-03-24,COST,New Excel Peter Lynch Chart Template Is Released


Ticker Relevance Analysis
We checked whether articles mentioned their assigned ticker or company name in the headline or body. Of 40,651 articles, 39,945 (98.26%) contained an explicit match, while 706 (1.74%) had no match. We decided to retain all records because keyword matching alone cannot reliably determine relevance; further sentence-level filtering will be performed before FinBERT classification.

# Financial News Preprocessing — Summary

**Project:** AI Stock Analyzer (LSTM–GRU + FinBERT + Mistral 7B)

- **Dataset:** 49,921 news records (2015–2023) for COST, MRK, NVDA, OXY, SPY and WFC.
- **Missing data:** Removed 9,124 records without article bodies; retained **40,797**. Dropped `author` and `publisher` (unusable/missing) and `source_domain` (constant for retained records).
- **Dates:** Standardized timestamps to `YYYY-MM-DD`; retained raw data separately. News will be aligned to the **next actual trading session** for forecasting to limit look-ahead bias.
- **Duplicates:** Removed 146 same-date, same-ticker, identical-body duplicates; **40,651 records remain**. Kept articles associated with multiple tickers.
- **Text cleaning:** Decoded HTML entities, stripped HTML tags, normalized Unicode and whitespace. **0 empty cleaned titles/articles; 0 remaining HTML-like tags.** Preserved original text and created `title_clean` / `article_clean`.
- **Relevance audit:** 13,769 title matches, 26,176 article-only matches, **706 without explicit ticker/company-name match**. These are approximate keyword checks; no articles were removed on this basis.

**Current status:** News cleaning and initial relevance audit complete. FinBERT inference has **not** started.

**Next session:** Segment long articles into sentences → inspect lengths → select ticker-relevant sentences/context windows → run FinBERT → aggregate sentiment and retain evidence for Mistral 7B.
